In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [2]:
file_paths = [
    "data/IMU/depth_5cm_motor_pow_0.csv",
    "data/IMU/depth_5cm_motor_pow_25.csv",
    "data/IMU/depth_5cm_motor_pow_50.csv",
    "data/IMU/depth_5cm_motor_pow_75.csv",
    "data/IMU/depth_5cm_motor_pow_100.csv",
    "data/IMU/depth_5cm_motor_pow_0_100.csv",

    "data/IMU/depth_10cm_motor_pow_0_100.csv",
    "data/IMU/depth_10cm_motor_pow_0.csv",
    "data/IMU/depth_10cm_motor_pow_25.csv",
    "data/IMU/depth_10cm_motor_pow_50.csv",
    "data/IMU/depth_10cm_motor_pow_75.csv",
    "data/IMU/depth_10cm_motor_pow_100.csv",
    
    "data/IMU/depth_15cm_motor_pow_0_100.csv",
    "data/IMU/depth_15cm_motor_pow_0.csv",
    "data/IMU/depth_15cm_motor_pow_25.csv",
    "data/IMU/depth_15cm_motor_pow_50.csv",
    "data/IMU/depth_15cm_motor_pow_75.csv",
    "data/IMU/depth_15cm_motor_pow_100.csv",
]

In [5]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ==========================================
# --- DYNAMIC SLICING SETTINGS ---
# ==========================================
full = False            # Set to True to process the entire file
start_row = 1369        # Which row to start processing from (if full=False)
num_rows = 685          # How many rows to plot (if full=False)
sampling_rate = 136.8   # 136.8 rows = 1 second
# ==========================================

# 1. Separate and organize paths into clean 5-element arrays (excluding '0_100')
depths = ["5cm", "10cm", "15cm"]
power_suffixes = ["pow_0.csv", "pow_25.csv", "pow_50.csv", "pow_75.csv", "pow_100.csv"]
labels = ["Power 0%", "Power 25%", "Power 50%", "Power 75%", "Power 100%"]
colors = plt.cm.plasma(np.linspace(0, 0.85, 5))

# (Assuming file_paths is defined earlier in your script as a list of strings)
grouped_paths = {}
for depth in depths:
    grouped_paths[depth] = [
        next(fp for fp in file_paths if f"depth_{depth}" in fp and fp.endswith(suffix))
        for suffix in power_suffixes
    ]

# 2. Process each depth group and generate the stacked plots
output_base_dir = "../../Simulations/IMU/Radar_20_june"
os.makedirs(output_base_dir, exist_ok=True)

for depth, paths in grouped_paths.items():
    fig, axes = plt.subplots(5, 1, figsize=(10, 11), sharex=True, sharey=True)
    
    status_msg = "FULL FILE" if full else f"ROWS {start_row} to {start_row + num_rows}"
    print(f"Generating stacked plot for depth: {depth} | Scope: {status_msg}...")

    for i, fp in enumerate(paths):
        file_path = f"../../{fp}"
        df_full = pd.read_csv(file_path)
        
        # Apply the dynamic row slicing logic
        if not full:
            # Ensure we don't go out of bounds
            actual_end_row = min(start_row + num_rows, len(df_full))
            df_target = df_full.iloc[start_row:actual_end_row].copy()
        else:
            df_target = df_full.copy()

        # Generate a time axis based strictly on the sampling rate, starting at 0
        num_target_rows = len(df_target)
        time_axis = np.arange(num_target_rows) / sampling_rate

        # Process the targeted slice
        az_raw = df_target['az'].to_numpy()
        az_corrected = az_raw + 9.80
        az_centered = az_corrected - np.mean(az_corrected)
        
        ax = axes[i]
        
        # Plotting details
        ax.plot(time_axis, az_centered, color=colors[i], linewidth=1, label=labels[i])
        ax.set_ylabel(r"$a_z$ ($\text{m/s}^2$)", fontsize=10)
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='none')
        
        # Force the plot to start exactly at 0 and end exactly at the max time (removes left/right empty gaps)
        if num_target_rows > 0:
            ax.set_xlim(0, time_axis[-1])
        
        # Style cleanup
        for spine in ["top", "right"]:
            ax.spines[spine].set_visible(False)

    axes[-1].set_xlabel(r'Time ($s$)', fontsize=11)
    
    # Update title dynamically
    title_suffix = "(Full File)" if full else f"(Rows {start_row}-{start_row + num_rows})"
    fig.suptitle(f'Linear Vertical Acceleration Stacked Comparison — Depth: {depth}\n{title_suffix}', 
                 fontsize=13, fontweight='bold', y=0.98)
    
    plt.tight_layout()
    
    # Save the file
    save_tag = "full" if full else f"slice_{start_row}_{start_row+num_rows}"
    plt.savefig(os.path.join(output_base_dir, f"stacked_IMU_depth_{depth}_{save_tag}.png"), dpi=300)
    plt.close()

print("All depth stacked plots completed.")

Generating stacked plot for depth: 5cm | Scope: ROWS 1369 to 2054...
Generating stacked plot for depth: 10cm | Scope: ROWS 1369 to 2054...
Generating stacked plot for depth: 15cm | Scope: ROWS 1369 to 2054...
All depth stacked plots completed.
